In [3]:
import sqlite3
import pandas as pd
from db_setup import setup_database
setup_database()
conn=sqlite3.connect('bookcycle.db')

✅ Database setup complete: Tables created and populated with data!


In [8]:
query="SELECT * FROM books LIMIT 5;"
query2="SELECT * FROM customers LIMIT 5;"
query3="SELECT * FROM transactions LIMIT 5;"
result=pd.read_sql(query,conn)
result2=pd.read_sql(query2,conn)
result3=pd.read_sql(query3,conn)
display(result, result2, result3)

,book_id,title,author,isbn,genre,condition,purchase_price,list_price,date_acquired,current_location,quantity
0,B1001,A Christmas Carol,Charles Dickens,9780141324524,Classic Fiction,Good,5.5,8.99,2023-01-15,Suburban,2
1,B1002,A Farewell to Arms,Ernest Hemingway,9780684801469,Classic Fiction,Very Good,7.0,11.99,2023-01-15,University,3
2,B1003,A Tale of Two Cities,Charles Dickens,9780141439600,Classic Fiction,Fair,4.5,7.99,2023-01-16,Downtown,2
3,B1004,Adventures of Huckleberry Finn,Mark Twain,9780142437179,Classic Fiction,Good,6.0,9.99,2023-01-16,University,4
4,B1005,Agnes Grey,Anne Bronte,9780140432107,Classic Fiction,Fair,4.0,7.99,2023-01-17,Suburban,1


,customer_id,join_date,is_member,zip_code,birth_year,preferred_store
0,C1001,2022-01-15,1,98105,1995,University
1,C1002,2022-01-15,0,98115,1988,Suburban
2,C1003,2022-01-16,1,98101,1992,Downtown
3,C1004,2022-01-16,1,98105,1999,University
4,C1005,2022-01-16,0,98115,1975,Suburban


,transaction_id,date_time,store_location,customer_id,book_id,sale_price,payment_method,is_online
0,T1001,2023-01-15 09:23:45,University,C1045,B1055,11.99,credit,0
1,T1002,2023-01-15 10:15:22,Downtown,C1023,B1032,10.99,debit,0
2,T1003,2023-01-15 10:45:33,Suburban,C1078,B1036,11.99,cash,0
3,T1004,2023-01-15 11:30:15,University,C1012,B1071,13.99,credit,1
4,T1005,2023-01-15 13:45:22,University,C1034,B1075,12.99,debit,0


In [14]:
# calculating total_sales from each location
query= """
WITH store_sales AS(
 SELECT store_location, SUM(sale_price) AS total_sales
 FROM transactions
 GROUP BY store_location
)
SELECT * FROM store_sales ORDER BY total_sales DESC;
"""
result=pd.read_sql(query,conn)
display(result)

,store_location,total_sales
0,University,586.50
1,Suburban,286.74
2,Downtown,241.76


In [27]:
# CTEs with Window Functions
# to find out most sold book from each category
query= """
WITH popularbooks AS(
SELECT b.title, b.genre, COUNT(*) as sold_copies,
ROW_NUMBER() OVER(
PARTITION BY b.genre
ORDER BY COUNT(*) DESC) AS genrewiseranked
FROM books b NATURAL JOIN transactions t
GROUP BY b.genre, b.title
)
SELECT title, genre, sold_copies FROM popularbooks WHERE genrewiseranked=1;
"""
result=pd.read_sql(query,conn)
display(result)

,title,genre,sold_copies
0,Hamlet,Classic Drama,4
1,The Wind in the Willows,Classic Fiction,5
2,The Adventures of Sherlock Holmes,Classic Mystery,4
3,Walden,Classic Non-Fiction,2
4,The Prince,Classic Philosophy,2
5,The Odyssey,Classic Poetry,4


In [40]:
query="SELECT b.genre, b.title, COUNT(*) FROM books b NATURAL JOIN transactions t GROUP BY b.title, b.genre;"
result=pd.read_sql(query,conn)
display(result)

,genre,title,COUNT(*)
0,Classic Fiction,A Christmas Carol,4
1,Classic Fiction,A Tale of Two Cities,4
2,Classic Fiction,Anne of Green Gables,3
3,Classic Fiction,Brave New World,3
4,Classic Drama,Hamlet,4
5,Classic Fiction,Invisible Man,1
6,Classic Fiction,Jane Eyre,4
7,Classic Fiction,Little Women,3
8,Classic Drama,Macbeth,2
9,Classic Fiction,Nineteen Eighty-Four,4


In [41]:
# to find out top selling book in each genre + their percentage of total genre sales

query = """
WITH book_sales AS (
    SELECT 
        b.book_id, 
        b.title, 
        b.genre, 
        COUNT(*) AS sales_count
    FROM books b
    JOIN transactions t ON b.book_id = t.book_id
    GROUP BY b.book_id
),
genre_totals AS (
    SELECT genre, SUM(sales_count) AS total_genre_sales
    FROM book_sales
    GROUP BY genre
),
ranked_books AS (
    SELECT 
        bs.genre,
        bs.title,
        bs.sales_count,
        ROW_NUMBER() OVER (PARTITION BY bs.genre ORDER BY bs.sales_count DESC) AS rank_in_genre,
        ROUND(bs.sales_count * 100.0 / gt.total_genre_sales, 2) AS percent_of_genre_sales
    FROM book_sales bs
    JOIN genre_totals gt ON bs.genre = gt.genre
)
SELECT * 
FROM ranked_books
WHERE rank_in_genre <= 3
ORDER BY genre, rank_in_genre;

"""

df = pd.read_sql_query(query, conn)
display(df)

,genre,title,sales_count,rank_in_genre,percent_of_genre_sales
0,Classic Drama,Hamlet,4,1,50.00
1,Classic Drama,Macbeth,2,2,25.00
2,Classic Drama,Othello,2,3,25.00
3,Classic Fiction,The Catcher in the Rye,5,1,6.85
4,Classic Fiction,The Scarlet Letter,5,2,6.85
5,Classic Fiction,The Wind in the Willows,5,3,6.85
6,Classic Mystery,The Adventures of Sherlock Holmes,4,1,57.14
7,Classic Mystery,The Hound of the Baskervilles,3,2,42.86
8,Classic Non-Fiction,Walden,2,1,100.00
9,Classic Philosophy,The Prince,2,1,66.67
